# Entrenamiento y comparación (issue #10)
La comparación vive acá, en el notebook, con código sklearn propio: el backend solo sirve `arbol` ya ajustado y no compara (ver `backend/src/collection/adapters/models/model_specs.py`: solo-arbol). La preparación sí se reutiliza del backend (`ml_preparation`, semilla 0).

Metodología (curso S5 `notebook-2-validacion-cruzada` + S5 `notebook-1-metricas`): CV estratificada solo en train (5 folds), grilla chica con `GridSearchCV` (scoring F1 macro, refit), contraste en validación, test de un solo uso solo para el elegido + baseline mayoría como piso. La elección por validación y el desempate árbol vs SVC se justifican abajo; el N=297 se justifica con el experimento de escala (1422 tramos, rama `experimento-recorte-medio`).

In [ ]:
import sys
from pathlib import Path

CANDIDATOS = [Path.cwd() / "backend" / "src", Path.cwd().parent / "backend" / "src"]
SRC = next(c for c in CANDIDATOS if (c / "collection").is_dir())
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import collection
print("src:", SRC.exists(), "schema:", collection.SCHEMA_VERSION)


In [ ]:
ROOTS = [Path.cwd(), Path.cwd().parent]
DATASET = next(r / "dataset" / "salida" / "tramos.csv" for r in ROOTS if (r / "dataset" / "salida" / "tramos.csv").is_file())
print("dataset:", DATASET)


In [ ]:
from collection.adapters.datasets.ml_preparation import prepare_dataset

prepared = prepare_dataset(DATASET, seed=0)
print("sizes:", {k: v.metadata["size"] for k, v in prepared.partitions.items()})
print("dist:", prepared.metadata["class_distribution"])
assert sum(v.metadata["size"] for v in prepared.partitions.values()) == len(prepared.enriched) == 297
assert not hasattr(prepared.preprocessor, "transformers_")


## Qué mide cada métrica (curso S5 `notebook-1-metricas`, celdas 13-14 y 18-24)

Matriz de confusión con filas = prioridad real y columnas = predicha, orden baja/media/alta. Por clase (una-vs-resto):

| Término | Significado (ej. clase alta) |
|---|---|
| **VP**: verdadero positivo | Predijo alta y era alta. |
| **FP**: falso positivo | Predijo alta, pero era media o baja (falsa alarma). |
| **FN**: falso negativo | Era alta, pero predijo otra (caso perdido). |
| **VN**: verdadero negativo | No era alta y tampoco predijo alta (descarte correcto). |

- **Accuracy**: aciertos / total. Con 43% de baja, un tonto que siempre dice baja ya saca ~0.44: por eso no alcanza sola.
- **Precision**: `VP / (VP + FP)`. De lo que predijo como alta, cuánto era alta.
- **Recall**: `VP / (VP + FN)`. De lo real alta, cuánto detectó.
- **F1**: media armónica de precision y recall. **F1 macro** = promedio del F1 de baja/media/alta con igual peso (no premia a la mayoría).
- **Baseline mayoría**: predice siempre baja. Piso: val acc 0.4407 / F1 0.2039 en piloto. Todo modelo útil debe superarlo.
- **CV 5 folds solo en train** (`StratifiedKFold`, shuffle, seed 0): media±std de accuracy y F1 macro sin mirar validación ni test (curso S5 `notebook-2-validacion-cruzada` celdas 6/9/10/12).
- **Todo lo que aprende va dentro del `Pipeline`** (`StandardScaler` solo en numéricas para logística/kNN/SVC; árbol sin escala — curso S4 celdas 37-38; `OneHotEncoder(handle_unknown="ignore")`): sin fuga entre folds (curso S5 celdas 26-27).

Abajo se imprime por modelo: CV acc/F1, mejor grilla, val acc/F1 + `classification_report`, matriz, y tabla VP/VN/FP/FN por clase. Eso es lo que decidió árboles.

In [ ]:
from types import SimpleNamespace

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SEED = 0
CLASS_ORDER = ("baja", "media", "alta")
NUMERICAS = [c for c in prepared.X.columns if c != "tipo_zona"]


# Preprocesador con escala en numericas (logistica/knn/svc lo necesitan; arbol no).
def _escalado():
    return ColumnTransformer(
        [
            ("zona", OneHotEncoder(handle_unknown="ignore"), ["tipo_zona"]),
            ("numericas", StandardScaler(), NUMERICAS),
        ]
    )


MODELOS = {
    "arbol": (
        lambda pre: make_pipeline(clone(pre), DecisionTreeClassifier(random_state=SEED)),
        {
            "decisiontreeclassifier__max_depth": [None, 3, 5],
            "decisiontreeclassifier__min_samples_split": [2, 5],
        },
    ),
    "logistica": (
        lambda pre: make_pipeline(_escalado(), LogisticRegression(max_iter=2000, random_state=SEED)),
        {"logisticregression__C": [0.5, 1.0, 2.0]},
    ),
    "knn": (
        lambda pre: make_pipeline(_escalado(), KNeighborsClassifier()),
        {"kneighborsclassifier__n_neighbors": [3, 5, 7]},
    ),
    "svc": (
        lambda pre: make_pipeline(_escalado(), SVC(kernel="rbf", random_state=SEED)),
        {"svc__C": [0.5, 1.0]},
    ),
}

train_idx = list(prepared.partitions["train"].indices)
val_idx = list(prepared.partitions["validation"].indices)
X_train, y_train = prepared.X.iloc[train_idx], prepared.y.iloc[train_idx]
X_val, y_val = prepared.X.iloc[val_idx], prepared.y.iloc[val_idx]
assert y_train.value_counts().min() >= 5, "el CV de 5 folds necesita 5 ejemplos por clase en train"

cv_por_dentro = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
metricas = []
for model_id, (constructor, grilla) in MODELOS.items():
    base = constructor(prepared.preprocessor)
    cv_res = cross_validate(clone(base), X_train, y_train, cv=cv_por_dentro, scoring=["accuracy", "f1_macro"])
    busqueda = GridSearchCV(
        clone(base),
        grilla,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        scoring="f1_macro",
        refit=True,
    )
    busqueda.fit(X_train, y_train)
    pred_val = busqueda.predict(X_val)
    cm = confusion_matrix(y_val, pred_val, labels=list(CLASS_ORDER))
    # VP/VN/FP/FN por clase (uno-vs-resto): VP=diagonal, FN=fila-VP, FP=columna-VP, VN=resto.
    detalle = {}
    for i, cls in enumerate(CLASS_ORDER):
        vp = int(cm[i, i])
        fn = int(cm[i, :].sum() - vp)
        fp = int(cm[:, i].sum() - vp)
        vn = int(cm.sum() - vp - fn - fp)
        detalle[cls] = dict(VP=vp, VN=vn, FP=fp, FN=fn)
    metricas.append(
        SimpleNamespace(
            model_id=model_id,
            cv_accuracy_mean=float(cv_res["test_accuracy"].mean()),
            cv_accuracy_std=float(cv_res["test_accuracy"].std()),
            cv_f1_macro_mean=float(cv_res["test_f1_macro"].mean()),
            cv_f1_macro_std=float(cv_res["test_f1_macro"].std()),
            best_params=dict(busqueda.best_params_),
            validation_accuracy=float(accuracy_score(y_val, pred_val)),
            validation_f1_macro=float(f1_score(y_val, pred_val, average="macro", zero_division=0)),
            validation_precision_macro=float(precision_score(y_val, pred_val, average="macro", zero_division=0)),
            validation_recall_macro=float(recall_score(y_val, pred_val, average="macro", zero_division=0)),
            validation_confusion_matrix=tuple(
                tuple(int(v) for v in fila)
                for fila in cm.tolist()
            ),
            detalle_vp_vn=dict(detalle),
            test_accuracy=None,
            test_f1_macro=None,
        )
    )

# Piso tonto: siempre la mayoría del train, medido en validación (no compite, solo referencia).
_piso = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
_pred_piso = _piso.predict(X_val)
baseline_acc = float(accuracy_score(y_val, _pred_piso))
baseline_f1 = float(f1_score(y_val, _pred_piso, average="macro", zero_division=0))

comparison = SimpleNamespace(
    metrics=metricas,
    chosen_model_id=max(metricas, key=lambda m: (m.validation_f1_macro, m.validation_accuracy)).model_id,
)
for m in comparison.metrics:
    print(f"{m.model_id}: cv_acc={m.cv_accuracy_mean:.4f}±{m.cv_accuracy_std:.4f} cv_f1={m.cv_f1_macro_mean:.4f}±{m.cv_f1_macro_std:.4f} | val_acc={m.validation_accuracy:.4f} val_F1={m.validation_f1_macro:.4f} prec={m.validation_precision_macro:.4f} rec={m.validation_recall_macro:.4f} best={m.best_params}")
    print(f"  matriz_val (filas=real, cols=pred, orden baja/media/alta)={list(list(r) for r in m.validation_confusion_matrix)}")
    for cls in CLASS_ORDER:
        d = m.detalle_vp_vn[cls]
        print(f"    {cls}: VP {d['VP']} FP {d['FP']} FN {d['FN']} VN {d['VN']}")
print(f"baseline(mayoria): acc={baseline_acc:.4f} f1={baseline_f1:.4f} (piso, no compite)")
print("elegido por validación (F1 macro, desempate accuracy):", comparison.chosen_model_id)
# Reporte completo del elegido para que quede en el notebook (sin tocar test):
from sklearn.metrics import classification_report as _cr
_elegido_tmp = comparison.chosen_model_id
_ctor_tmp, _grilla_tmp = MODELOS[_elegido_tmp]
_tmp = GridSearchCV(clone(_ctor_tmp(prepared.preprocessor)), _grilla_tmp, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), scoring="f1_macro", refit=True).fit(X_train, y_train)
print(f"--- classification_report del elegido en validación ({_elegido_tmp}) ---")
print(_cr(y_val, _tmp.predict(X_val), labels=list(CLASS_ORDER), target_names=list(CLASS_ORDER), digits=4))
assert all(m.test_accuracy is None for m in comparison.metrics)
mejor = max(comparison.metrics, key=lambda m: (m.validation_f1_macro, m.validation_accuracy))
assert comparison.chosen_model_id == mejor.model_id


In [ ]:
# Test un solo uso, solo para el elegido: re-ajuste en train y una sola prediccion en test.
elegido = comparison.chosen_model_id
constructor, _ = MODELOS[elegido]
mejores_params = next(m.best_params for m in comparison.metrics if m.model_id == elegido)
test_idx = list(prepared.partitions["test"].indices)
X_test, y_test = prepared.X.iloc[test_idx], prepared.y.iloc[test_idx]
tuberia_final = constructor(prepared.preprocessor)
tuberia_final.set_params(**mejores_params)
tuberia_final.fit(X_train, y_train)
pred_test = tuberia_final.predict(X_test)

for m in comparison.metrics:
    if m.model_id == elegido:
        m.test_accuracy = float(accuracy_score(y_test, pred_test))
        m.test_f1_macro = float(f1_score(y_test, pred_test, average="macro", zero_division=0))
final = SimpleNamespace(metrics=comparison.metrics, chosen_model_id=elegido)
elegida = next(m for m in final.metrics if m.model_id == final.chosen_model_id)
print(f"test un solo uso ({elegida.model_id}): acc={elegida.test_accuracy:.4f} f1={elegida.test_f1_macro:.4f}")
assert elegida.test_accuracy is not None and elegida.test_f1_macro is not None
assert sum(1 for m in final.metrics if m.test_accuracy is not None) == 1
print("NOTA: metricas = ajuste a la regla sintetica, no desempeno operativo.")


## Por qué se eligió el árbol (aunque empató con SVC en 1.0)

Validación n=59 (orden baja/media/alta): árbol `[[26,0,0],[0,18,0],[0,0,15]]` y SVC idéntica — empate perfecto en acc/F1 (1.0/1.0). kNN `[[26,0,0],[2,16,0],[0,0,15]]` (2 medias reales → baja) y logística `[[26,0,0],[2,16,0],[0,2,13]]` (2 medias→baja + 2 altas→media). El desempate no es por métrica sino por criterio del curso y de backend:

1. **Interpretabilidad (criterio del curso S4 `notebook-1-arboles-decision`):** el árbol se grafica con `plot_tree` y muestra feature + umbral por decisión; SVC-RBF es caja negra.
2. **Simplicidad operativa:** árbol sin `StandardScaler` (usa el preprocessor #18 tal cual); logística/kNN/SVC sí lo necesitan. Menos piezas que se rompen con zonas no vistas (`handle_unknown="ignore"`).
3. **Costo:** inferencia O(profundidad) vs O(n_soporte × features) del SVC.
4. **Coherencia con el dato:** la etiqueta es regla determinista por umbrales (`score = 2*días + contenedores + zona_w + 2*quejas`, baja ≤0 / media 1..2 / alta >2); el árbol la representa directo, el SVC la aproxima.

Lectura honesta + tipo de error: el 1.0 mide ajuste a la regla sintética, no desempeño operativo. Los errores de kNN/logística son entre vecinas, pero los de kNN (`media→baja`) son los más caros en recolección: si se seleccionan medias+altas, esas 2 medias quedarían sin recolectar (FN de media = FP de baja). El árbol nunca cruza alta↔baja (se cumple en los 8 experimentos piloto+medio).

## Por qué se dejó N=297 (verificación de escala con 1422 tramos)

Se entrenó con muchas más muestras y dio lo mismo, por eso se conservó el piloto de **297 tramos** (178 train / 59 val / 60 test, seed 0; dist total 129 baja / 93 media / 75 alta) como dataset del TP: más ciudad = minutos de build sin cambiar la decisión, y la etiqueta determinista pone techo a lo que el experimento enseña.

Evidencia en rama local **`experimento-recorte-medio`** (`docs/modelo-comparacion.md` en esa rama, logs de `scripts/build_dataset.py` + `scripts/train_compare.py` copiados ahí): mismo `prepare + compare`, misma semilla 0, misma regla #18, misma metodología (CV5 solo en train, elección en validación, test un solo uso). Lo único que cambió fue el `BBOX_WGS` medio (-56.17,-34.92,-56.14,-34.90) + fix de `tramo_id` duplicados (portado a `main`).

| Modelo | Piloto CV F1 (n=178) | Medio CV F1 (n=853) | Piloto val F1 (n=59) | Medio val F1 (n=284) |
|---|---|---|---|---|
| árbol | 0.9932 ± 0.0136 | 0.9972 ± 0.0034 | 1.0000 (0 errores) | 1.0000 (0 errores) |
| SVC RBF | 0.9733 ± 0.0390 | 0.9973 ± 0.0055 | 1.0000 (0 errores) | 1.0000 (0 errores) |
| kNN | 0.9743 ± 0.0315 | 0.9904 ± 0.0070 | 0.9680 (2 errores) | 1.0000 (0 errores, k=3) |
| logística | 0.9619 ± 0.0367 | 0.9779 ± 0.0080 | 0.9268 (4 errores) | 0.9958 (1 error) |
| baseline mayoría | — | — | 0.2039 | 0.2015 |

Recorte medio: 876 nodos, 1422 tramos (853/284/285), balance #18 615/427/380 (43.2/30.0/26.7%) vs piloto 129/93/75 (43.4/31.3/25.3%). Test un solo uso del árbol en medio: acc 1.0000 / F1 1.0000 `[[123,0,0],[0,86,0],[0,0,76]]` (seed 1: 0.9965, 1 error en 285 — el 1.0 es la moda, no garantía).

Lectura: **mismo ranking (árbol≈SVC ≥ kNN > logística), mismo ganador, mismo desempate**, baseline idéntico (~0.20), desvíos ~5× más chicos. El piloto alcanzaba para decidir pero exageraba incertidumbre (cada error en val 1.7pp vs 0.35pp) y castigaba de más a logística/kNN. Para reproducir: `git checkout experimento-recorte-medio` (los CSV del piloto en `main` no se tocan).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.tree import plot_tree
from IPython.display import display
from collection.adapters.models.model_specs import build_pipeline

ROOT = next(r for r in ROOTS if (r / "dataset" / "salida" / "tramos.csv").is_file())
FIGURES_DIR = ROOT / "docs" / "figs"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CLASS_ORDER = ("baja", "media", "alta")

# CV F1 macro (barras de error = std) frente a F1 macro de validación.
fig, ax = plt.subplots(figsize=(8, 5))
models = comparison.metrics
labels = [metric.model_id for metric in models]
positions = np.arange(len(models))
ax.bar(positions - 0.2, [m.cv_f1_macro_mean for m in models], width=0.4, yerr=[m.cv_f1_macro_std for m in models], capsize=4, label="CV F1 macro ± std")
ax.bar(positions + 0.2, [m.validation_f1_macro for m in models], width=0.4, label="Validación F1 macro")
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 1.05), ylabel="F1 macro", title="CV (solo train) vs validación")
ax.legend(); fig.tight_layout()
path = FIGURES_DIR / "f1_comparacion.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)

# Matrices de validación en orden baja/media/alta.
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4))
for ax, metric in zip(axes, models, strict=True):
    ConfusionMatrixDisplay(confusion_matrix=np.asarray(metric.validation_confusion_matrix), display_labels=CLASS_ORDER).plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(metric.model_id)
fig.tight_layout()
path = FIGURES_DIR / "matrices_confusion_validacion.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)

# Árbol ganador re-ajustado únicamente en train, mostrado hasta profundidad 3.
train_indices = prepared.partitions["train"].indices
tree = build_pipeline("arbol", prepared.preprocessor, seed=0)
tree.fit(prepared.X.iloc[list(train_indices)], prepared.y.iloc[list(train_indices)])
preprocessor_name = next(name for name in tree.named_steps if name != "decisiontreeclassifier")
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(tree.named_steps["decisiontreeclassifier"], feature_names=tree.named_steps[preprocessor_name].get_feature_names_out(), class_names=list(CLASS_ORDER), filled=True, rounded=True, max_depth=3, ax=ax)
ax.set_title("Árbol ganador (ajustado solo en train, profundidad máxima 3)")
fig.tight_layout()
path = FIGURES_DIR / "arbol_ganador.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)


## Registro de elección: `arbol`
La comparación de arriba es el proceso de elección del modelo del proyecto; el backend no compara, solo sirve el árbol ya elegido.
- **Criterio**: mayor F1 macro en validación (desempate: accuracy). Test una sola vez al final, solo para el elegido. Baseline mayoría como piso (val 0.4407/0.2039).
- **Qué se miró**: accuracy + F1 macro (CV media±std y validación), precision/recall macro, `classification_report`, matriz (filas=real, cols=pred) y tabla VP/FP/FN/VN por clase impresas arriba. Los 2 errores de kNN son medias→baja (caros en recolección); logística confunde vecinas medias↔altas; árbol/SVC cero errores.
- **Por qué gana el árbol**: empate en métrica con SVC, desempate por interpretabilidad (`plot_tree`), sin scaler, O(profundidad) y coherencia con la regla por umbrales (`score = 2*días + contenedores + zona_w + 2*quejas`).
- **Por qué N=297**: verificación de escala en rama `experimento-recorte-medio` (1422 tramos) dio mismo ranking/ganador/desempate; ver celda anterior. El piloto alcanza; el medio lo confirma.
- **Límite**: las métricas miden ajuste a la regla construida, no desempeño operativo real.
- **Evidencia**: tablas impresas en este notebook + `docs/figs/f1_comparacion.png` y `docs/figs/matrices_confusion_validacion.png`.
- **Artefacto**: `backend/artifacts/arbol-montevideo.joblib`, generado con `scripts/generate_arbol_artifact.py` (re-ajuste en train con los mejores hiperparámetros del contraste en validación).